# Poisson in 3D — the Same Weak Form on Tetrahedra

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/poisson_3d.ipynb)

The point of this notebook is what *doesn't* change. Swap the rectangle for a
tetrahedral cube and the assembler, the condenser and the solve are byte-for-byte
the same as in the [2D notebook](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/poisson.ipynb):
$\nabla u \cdot \nabla v$ carries no hard-coded spatial dimension.

The result is rendered with pyvista as a half-domain cutaway — the solution
vanishes on the boundary, so the structure is only visible once the cube is
opened up.

🖥️ *Installs pyvista and a virtual framebuffer, so the first cell takes a
little longer than in the 2D notebooks.*

Docs: [3D extension](https://docs.tensor-mesh.com/example_gallery/poisson.html#d-extension-poisson-3d-py) · Source: [`examples/poisson/poisson_3d.py`](https://github.com/camlab-ethz/TensorMesh/blob/main/examples/poisson/poisson_3d.py)

In [ ]:
# Install TensorMesh (skipped automatically if it is already available, e.g. a local dev setup).
# The apt line provides the OpenGL utility library that gmsh -- TensorMesh's mesh generator --
# needs at import time; it is a no-op where the library is already present.
# pyvista renders the 3D figures; xvfb gives it a virtual display to draw into.
import importlib.util
if importlib.util.find_spec("tensormesh") is None:
    !apt-get -qq install -y libglu1-mesa libgl1-mesa-glx xvfb > /dev/null 2>&1 || true
    %pip install -q tensormesh-fem==0.2.0 pyvista

import contextlib
import os


@contextlib.contextmanager
def quiet():
    """Hide gmsh's meshing log, which is written below Python's stdout.
    Drop the ``with quiet():`` wrapper anywhere to see what the mesher is doing."""
    with open(os.devnull, "w") as null:
        saved = os.dup(1)
        os.dup2(null.fileno(), 1)
        try:
            yield
        finally:
            os.dup2(saved, 1)
            os.close(saved)

## Setup

`pv.OFF_SCREEN = True` renders into the virtual framebuffer rather than a
window; the VTK warning switch quiets the software-OpenGL fallback notices
that every headless machine produces.

In [ ]:
import warnings

import pyvista as pv
import vtk

vtk.vtkObject.GlobalWarningDisplayOff()          # software-OpenGL fallback notices
warnings.filterwarnings("ignore", message=".*Use vtk with osmesa.*")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="pyvista.*")
pv.OFF_SCREEN = True

import torch

from tensormesh import (Condenser, LaplaceElementAssembler, MassElementAssembler,
                        Mesh, NodeAssembler)


class LoadAssembler(NodeAssembler):
    """Assemble the load vector f_i = int f v_i dx."""

    def forward(self, v, f):
        return v * f

## Solve

In [ ]:
CHARA_LENGTH = 0.06   # lower for a finer mesh (and a slower solve)

with quiet():
    mesh = Mesh.gen_cube(chara_length=CHARA_LENGTH).double()
x = mesh.points
print(f"mesh: {mesh.n_points} nodes, {mesh.n_elements} tetrahedra")

# Manufactured solution u = sin(pi x) sin(pi y) sin(pi z), zero on the boundary,
# so that -Laplace(u) = 3 pi^2 u.
pi = torch.pi
u_exact = torch.sin(pi * x[:, 0]) * torch.sin(pi * x[:, 1]) * torch.sin(pi * x[:, 2])
f = 3.0 * pi ** 2 * u_exact

# The very same assembler as the 2D notebook: grad(u) . grad(v) has no hard-coded
# spatial dimension, so nothing about the weak form changes in 3D.
K = LaplaceElementAssembler.from_mesh(mesh)(mesh.points)
rhs = LoadAssembler.from_mesh(mesh)(mesh.points, point_data={"f": f})

condenser = Condenser(mesh.boundary_mask, torch.zeros_like(u_exact))
K_, rhs_ = condenser(K, rhs)
u = condenser.recover(K_.solve(rhs_))

# Mass-weighted relative L2 error.
e = u - u_exact
M = MassElementAssembler.from_mesh(mesh)(mesh.points)
rel_l2 = (torch.sqrt((e * (M @ e)).sum()) / torch.sqrt((u_exact * (M @ u_exact)).sum())).item()
print(f"relative L2 error: {rel_l2:.3e}")

## Render

`mesh.save` writes a standard `.vtu`, which is also what you would open in
ParaView — pyvista reads the same file here.

In [ ]:
# Attach the fields and hand the mesh to pyvista for a half-domain cutaway:
# the solution is ~0 on the boundary, so the interesting structure only shows
# once the cube is opened up.
mesh.register_point_data("u_fem", u)
mesh.register_point_data("u_exact", u_exact)
mesh.register_point_data("error", e)
mesh.save("poisson_3d.vtu")

grid = pv.read("poisson_3d.vtu")
kept = grid.clip(normal="x", origin=(0.5, 0.5, 0.5),
                 invert=True).extract_surface(algorithm="dataset_surface")

plotter = pv.Plotter(off_screen=True, window_size=(1100, 750))
plotter.add_mesh(grid.extract_surface(algorithm="dataset_surface"),
                 color="gray", opacity=0.35,
                 style="wireframe", line_width=1)
plotter.add_mesh(kept, scalars="u_fem", cmap="turbo", smooth_shading=True,
                 clim=(float(kept["u_fem"].min()), float(kept["u_fem"].max())))
plotter.add_title("3D Poisson — half-domain cut at x = 0.5")
plotter.camera_position = [(3, 3, 3), (0.5, 0.5, 0.5), (0.0, 0.0, 1.0)]
plotter.screenshot("poisson_3d.png")
plotter.close()

from IPython.display import Image
Image("poisson_3d.png")

## Where to next

- [Cantilever beam](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/cantilever_beam.ipynb) — 3D linear elasticity with a vector-valued field.
- [h-adaptivity](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/poisson_h_adaptivity.ipynb) — spend elements where the error actually is.